# License Plate Detection — Full Pipeline
**CMPS 261 — Machine Learning Project**

This notebook runs the **complete pipeline** in order:
1. Exploratory Data Analysis
2. YOLOv8s Training & Evaluation
3. Faster R-CNN Training & Evaluation
4. RetinaNet Training & Evaluation
5. Final Comparison

**Smart skip:** If trained weights already exist in `models/`, training is skipped automatically — the notebook loads the weights and jumps straight to evaluation. No need to retrain.

**Runs locally and on Google Colab** — environment is detected automatically.

## Environment Setup
Run this first — sets all paths and installs dependencies based on whether you are on Colab or local.

In [ ]:
import sys, os, json, random, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from tqdm import tqdm
import torch
import os, certifi
os.environ.setdefault('SSL_CERT_FILE', certifi.where())
os.environ.setdefault('REQUESTS_CA_BUNDLE', certifi.where())

IN_COLAB = 'google.colab' in sys.modules
print(f'Running on: {"Google Colab" if IN_COLAB else "Local"}')

if IN_COLAB:
    import subprocess
    subprocess.run(['pip', 'install', 'ultralytics', 'pycocotools', '-q'], check=True)

    from google.colab import drive
    drive.mount('/content/drive')
    import zipfile
    zip_path = '/content/drive/MyDrive/license_plate_data.zip'
    if not os.path.exists('/content/data/yolo'):
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall('/content/')
        print('Dataset extracted.')
    # Verify extraction produced what we need
    for d in ['/content/data/archive/images', '/content/data/archive/annotations']:
        if not os.path.isdir(d):
            raise RuntimeError(f'Extraction did not produce {d}. Check the zip on Drive.')

    IMG_DIR   = '/content/data/archive/images'
    ANN_DIR   = '/content/data/archive/annotations'
    TRAIN_IMG = '/content/data/yolo/images/train'
    VAL_IMG   = '/content/data/yolo/images/val'
    TEST_IMG  = '/content/data/yolo/images/test'
    TRAIN_LBL = '/content/data/yolo/labels/train'
    VAL_LBL   = '/content/data/yolo/labels/val'
    TEST_LBL  = '/content/data/yolo/labels/test'
    MODELS_DIR    = '/content/models'
    RESULTS_DIR   = '/content/results'
    YOLO_WEIGHTS  = '/content/models/yolov8s_best.pt'
    FRCNN_WEIGHTS = '/content/models/fasterrcnn_best.pth'
    RETINA_WEIGHTS= '/content/models/retinanet_best.pth'

    yaml_content = """path: /content/data/yolo
train: images/train
val:   images/val
test:  images/test
nc: 1
names: ['licence']
"""
    os.makedirs('/content/data/yolo', exist_ok=True)
    with open('/content/data/yolo/dataset.yaml', 'w') as f:
        f.write(yaml_content)
    YAML_PATH = '/content/data/yolo/dataset.yaml'

else:
    sys.path.append('..')
    IMG_DIR   = '../data/archive/images'
    ANN_DIR   = '../data/archive/annotations'
    TRAIN_IMG = '../data/yolo/images/train'
    VAL_IMG   = '../data/yolo/images/val'
    TEST_IMG  = '../data/yolo/images/test'
    TRAIN_LBL = '../data/yolo/labels/train'
    VAL_LBL   = '../data/yolo/labels/val'
    TEST_LBL  = '../data/yolo/labels/test'
    MODELS_DIR    = '../models'
    RESULTS_DIR   = '../results'
    YOLO_WEIGHTS  = '../models/yolov8s_best.pt'
    FRCNN_WEIGHTS = '../models/fasterrcnn_best.pth'
    RETINA_WEIGHTS= '../models/retinanet_best.pth'

    from src.prepare_data import prepare
    YAML_PATH = prepare()

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

DEVICE = (torch.device('cuda') if torch.cuda.is_available() else
          torch.device('mps')  if torch.backends.mps.is_available() else
          torch.device('cpu'))

print(f'Device      : {DEVICE}' + (f'  ({torch.cuda.get_device_name(0)})' if torch.cuda.is_available() else ''))
print(f'YAML        : {YAML_PATH}')
print(f'Models dir  : {MODELS_DIR}')
print(f'Results dir : {RESULTS_DIR}')
print(f'\nWeights status:')
for name, path in [('YOLOv8s', YOLO_WEIGHTS), ('Faster R-CNN', FRCNN_WEIGHTS), ('RetinaNet', RETINA_WEIGHTS)]:
    print(f'  {name:15} : {"FOUND — will skip training" if os.path.exists(path) else "NOT FOUND — will train"}')

---
# Part 1 — Exploratory Data Analysis
Parse all annotations, visualise image sizes, bounding box distributions, and sample images.

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd

records = []
for fname in sorted(os.listdir(ANN_DIR)):
    if not fname.endswith('.xml'): continue
    tree = ET.parse(os.path.join(ANN_DIR, fname))
    root = tree.getroot()
    img_w = int(root.find('size/width').text)
    img_h = int(root.find('size/height').text)
    for obj in root.findall('object'):
        bb = obj.find('bndbox')
        records.append({
            'filename': root.find('filename').text,
            'img_w': img_w, 'img_h': img_h,
            'xmin': int(bb.find('xmin').text), 'ymin': int(bb.find('ymin').text),
            'xmax': int(bb.find('xmax').text), 'ymax': int(bb.find('ymax').text),
        })

df = pd.DataFrame(records)
df['box_w']      = df['xmax'] - df['xmin']
df['box_h']      = df['ymax'] - df['ymin']
df['box_area']   = df['box_w'] * df['box_h']
df['img_area']   = df['img_w'] * df['img_h']
df['area_ratio'] = df['box_area'] / df['img_area']
df['aspect']     = df['box_w'] / df['box_h']
print(f'Total annotations : {len(df)}')
print(f'Unique images     : {df["filename"].nunique()}')
print(df[['img_w','img_h','box_w','box_h','area_ratio']].describe().round(3))

### Image Size Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['img_w'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Image Width Distribution'); axes[0].set_xlabel('Width (px)')
axes[1].hist(df['img_h'], bins=30, color='coral',     edgecolor='white')
axes[1].set_title('Image Height Distribution'); axes[1].set_xlabel('Height (px)')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'image_size_distribution.png'), dpi=150)
plt.show()

### Bounding Box Size & Aspect Ratio

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df['box_w'],      bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Box Width');  axes[0].set_xlabel('px')
axes[1].hist(df['box_h'],      bins=30, color='coral',     edgecolor='white')
axes[1].set_title('Box Height'); axes[1].set_xlabel('px')
axes[2].hist(df['aspect'],     bins=30, color='green',     edgecolor='white')
axes[2].set_title('Aspect Ratio (w/h)'); axes[2].set_xlabel('ratio')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'bbox_distributions.png'), dpi=150)
plt.show()

### Bounding Box Center Heatmap

In [ ]:
cx = ((df['xmin'] + df['xmax']) / 2) / df['img_w']
cy = ((df['ymin'] + df['ymax']) / 2) / df['img_h']
plt.figure(figsize=(6, 5))
plt.hist2d(cx, cy, bins=30, cmap='hot')
plt.colorbar(label='Count')
plt.title('Bounding Box Centre Heatmap (normalised)')
plt.xlabel('x (normalised)'); plt.ylabel('y (normalised)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'bbox_center_heatmap.png'), dpi=150)
plt.show()

### Sample Images with Bounding Boxes

In [ ]:
sample_files = random.sample(df['filename'].unique().tolist(), min(8, df['filename'].nunique()))
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for ax, fname in zip(axes, sample_files):
    img_path = os.path.join(IMG_DIR, fname)
    if not os.path.exists(img_path): ax.axis('off'); continue
    img = Image.open(img_path).convert('RGB')
    ax.imshow(img)
    rows = df[df['filename'] == fname]
    for _, row in rows.iterrows():
        ax.add_patch(patches.Rectangle(
            (row['xmin'], row['ymin']), row['box_w'], row['box_h'],
            linewidth=2, edgecolor='lime', facecolor='none'))
    ax.set_title(fname, fontsize=7); ax.axis('off')
plt.suptitle('Sample Images with Ground-Truth Boxes', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'sample_images.png'), dpi=150)
plt.show()

### Dataset Summary

In [ ]:
print('=' * 45)
print(f'  Total images      : {df["filename"].nunique()}')
print(f'  Total annotations : {len(df)}')
print(f'  Avg box width     : {df["box_w"].mean():.1f} px')
print(f'  Avg box height    : {df["box_h"].mean():.1f} px')
print(f'  Avg plate/image   : {df["area_ratio"].mean()*100:.1f}% of image area')
print(f'  Train / Val / Test: 303 / 64 / 66')
print('=' * 45)

---
# Shared Utilities
The dataset class and helper functions below are shared across all three models.

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

class LicensePlateDataset(Dataset):
    """Reads YOLO .txt labels, converts to xyxy absolute coords.
    augment=True applies random flip, colour jitter, scale and crop (training only)."""
    def __init__(self, img_dir, lbl_dir, augment=False):
        self.augment = augment
        self.samples = []
        for lbl_file in sorted(os.listdir(lbl_dir)):
            if not lbl_file.endswith('.txt'): continue
            stem = os.path.splitext(lbl_file)[0]
            for ext in ['.jpg', '.jpeg', '.png']:
                img_path = os.path.join(img_dir, stem + ext)
                if os.path.exists(img_path):
                    self.samples.append((img_path, os.path.join(lbl_dir, lbl_file)))
                    break

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, lbl_path = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        W, H = img.size
        boxes = []
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                _, cx, cy, w, h = map(float, parts[:5])
                xmin = max(0.0, (cx-w/2)*W);  ymin = max(0.0, (cy-h/2)*H)
                xmax = min(float(W),(cx+w/2)*W); ymax = min(float(H),(cy+h/2)*H)
                if xmax > xmin and ymax > ymin: boxes.append([xmin,ymin,xmax,ymax])
        if not boxes: boxes = [[0.0,0.0,1.0,1.0]]

        if self.augment:
            if random.random() > 0.5:
                img = TF.hflip(img); W2 = img.size[0]
                boxes = [[W2-b[2],b[1],W2-b[0],b[3]] for b in boxes]
            img = TF.adjust_brightness(img, 1+random.uniform(-0.4,0.4))
            img = TF.adjust_contrast(img,   1+random.uniform(-0.4,0.4))
            img = TF.adjust_saturation(img, 1+random.uniform(-0.3,0.3))
            img = TF.adjust_hue(img,        random.uniform(-0.08,0.08))
            if random.random() > 0.4:
                scale = random.uniform(0.75,1.0)
                new_W,new_H = int(W*scale),int(H*scale)
                img = TF.resize(img,(new_H,new_W))
                pad_x=random.randint(0,W-new_W); pad_y=random.randint(0,H-new_H)
                img = TF.pad(img,(pad_x,pad_y,W-new_W-pad_x,H-new_H-pad_y))
                boxes=[[b[0]*scale+pad_x,b[1]*scale+pad_y,b[2]*scale+pad_x,b[3]*scale+pad_y] for b in boxes]
            if random.random() > 0.5:
                before_img, before_boxes = img.copy(), [b[:] for b in boxes]
                crop_scale=random.uniform(0.85,1.0)
                cW,cH=int(W*crop_scale),int(H*crop_scale)
                x0=random.randint(0,W-cW); y0=random.randint(0,H-cH)
                img=TF.resize(TF.crop(img,y0,x0,cH,cW),(H,W))
                nb=[[max(0.0,(b[0]-x0)/crop_scale),max(0.0,(b[1]-y0)/crop_scale),
                     min(float(W),(b[2]-x0)/crop_scale),min(float(H),(b[3]-y0)/crop_scale)] for b in boxes]
                boxes=[b for b in nb if b[2]>b[0] and b[3]>b[1]]
                if not boxes:
                    img, boxes = before_img, before_boxes

        boxes  = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.ones(len(boxes), dtype=torch.int64)
        return TF.to_tensor(img), {'boxes':boxes,'labels':labels,
            'image_id':torch.tensor([idx]),
            'area':(boxes[:,3]-boxes[:,1])*(boxes[:,2]-boxes[:,0]),
            'iscrowd':torch.zeros(len(boxes),dtype=torch.int64)}

def collate_fn(batch): return tuple(zip(*batch))

def compute_iou(a, b):
    xA,yA=max(a[0],b[0]),max(a[1],b[1]); xB,yB=min(a[2],b[2]),min(a[3],b[3])
    inter=max(0,xB-xA)*max(0,yB-yA)
    return inter/((a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter+1e-6)

def collect_predictions(model_fn, loader, device):
    """Run the detector once and cache predictions/targets for threshold search."""
    cache=[]
    model_fn.eval()
    with torch.no_grad():
        for images, targets in tqdm(loader, desc='Collecting predictions', leave=False):
            images=[img.to(device) for img in images]
            preds=model_fn(images)
            for pred,target in zip(preds,targets):
                scores=pred['scores'].detach().cpu().numpy()
                order=np.argsort(-scores)
                cache.append({
                    'boxes': pred['boxes'].detach().cpu().numpy()[order],
                    'scores': scores[order],
                    'gt_boxes': target['boxes'].numpy(),
                })
    return cache

def evaluate_cached(cache, threshold):
    tp,fp,fn=0,0,0; iou_scores=[]
    for item in cache:
        gt_boxes=item['gt_boxes']
        pred_boxes=item['boxes'][item['scores']>=threshold]
        matched=set()
        for pb in pred_boxes:
            best_iou,best_j=0,-1
            for j,gb in enumerate(gt_boxes):
                if j in matched:
                    continue
                iou=compute_iou(pb,gb)
                if iou>best_iou: best_iou,best_j=iou,j
            if best_iou>=0.5 and best_j!=-1:
                tp+=1; matched.add(best_j); iou_scores.append(best_iou)
            else: fp+=1
        fn+=len(gt_boxes)-len(matched)
    p=tp/(tp+fp+1e-6); r=tp/(tp+fn+1e-6); f1=2*p*r/(p+r+1e-6)
    return p, r, f1, float(np.mean(iou_scores)) if iou_scores else 0.0

def find_best_threshold(cache, min_threshold=0.05, max_threshold=0.99):
    """Exact F1 search: F1 only changes when threshold crosses a prediction score."""
    score_arrays=[item['scores'] for item in cache if len(item['scores'])]
    if not score_arrays:
        return min_threshold, evaluate_cached(cache, min_threshold)
    scores=np.concatenate(score_arrays)
    candidates=np.unique(scores[(scores>=min_threshold) & (scores<=max_threshold)])
    candidates=np.unique(np.concatenate(([min_threshold, max_threshold], candidates)))
    best_thresh, best_metrics = min_threshold, evaluate_cached(cache, min_threshold)
    for thresh in candidates:
        metrics=evaluate_cached(cache, float(thresh))
        if (metrics[2], metrics[0], float(thresh)) > (best_metrics[2], best_metrics[0], best_thresh):
            best_thresh, best_metrics = float(thresh), metrics
    return best_thresh, best_metrics

def compute_metrics(model_fn, loader, threshold, device):
    return evaluate_cached(collect_predictions(model_fn, loader, device), threshold)

workers = 2 if IN_COLAB else 0
train_ds = LicensePlateDataset(TRAIN_IMG, TRAIN_LBL, augment=True)
val_ds   = LicensePlateDataset(VAL_IMG,   VAL_LBL,   augment=False)
test_ds  = LicensePlateDataset(TEST_IMG,  TEST_LBL,  augment=False)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
train_loader = DataLoader(train_ds,batch_size=4,shuffle=True, collate_fn=collate_fn,num_workers=workers)
val_loader   = DataLoader(val_ds,  batch_size=4,shuffle=False,collate_fn=collate_fn,num_workers=workers)
test_loader  = DataLoader(test_ds, batch_size=4,shuffle=False,collate_fn=collate_fn,num_workers=workers)

---
# Part 2 — Model 1: YOLOv8s
Single-stage anchor-free detector. Fastest and most accurate on this dataset.

In [ ]:
from ultralytics import YOLO
import shutil

if os.path.exists(YOLO_WEIGHTS):
    print(f'Weights found — skipping training, loading {YOLO_WEIGHTS}')
    yolo_model = YOLO(YOLO_WEIGHTS)
else:
    print('No weights found — training YOLOv8s...')
    yolo_model = YOLO('yolov8s.pt')
    yolo_model.train(
        data=YAML_PATH, epochs=100, imgsz=640, batch=16,
        device=0 if IN_COLAB else ('mps' if torch.backends.mps.is_available() else 'cpu'),
        project=MODELS_DIR, name='yolov8s', exist_ok=True, verbose=True,
    )
    src = os.path.join(MODELS_DIR, 'yolov8s', 'weights', 'best.pt')
    shutil.copy(src, YOLO_WEIGHTS)
    print(f'Saved: {YOLO_WEIGHTS}')

### YOLOv8s — Evaluate on Test Set

In [ ]:
test_metrics = yolo_model.val(split='test')
yolo_precision = test_metrics.box.mp
yolo_recall    = test_metrics.box.mr
yolo_f1        = 2*yolo_precision*yolo_recall/(yolo_precision+yolo_recall+1e-6)
yolo_map50     = test_metrics.box.map50
yolo_map50_95  = test_metrics.box.map

print(f'Precision    : {yolo_precision:.4f}')
print(f'Recall       : {yolo_recall:.4f}')
print(f'F1           : {yolo_f1:.4f}')
print(f'mAP@0.5      : {yolo_map50:.4f}')
print(f'mAP@0.5:0.95 : {yolo_map50_95:.4f}')

with open(os.path.join(RESULTS_DIR,'yolo_metrics.json'),'w') as f:
    json.dump({'model':'YOLOv8s','precision':round(yolo_precision,4),
               'recall':round(yolo_recall,4),'f1':round(yolo_f1,4),
               'map50':round(yolo_map50,4),'map50_95':round(yolo_map50_95,4)},f,indent=2)

### YOLOv8s — Sample Predictions

In [ ]:
test_images = random.sample(os.listdir(TEST_IMG), 8)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for ax, fname in zip(axes, test_images):
    img_path = os.path.join(TEST_IMG, fname)
    result   = yolo_model.predict(img_path, conf=0.25, verbose=False)[0]
    img      = Image.open(img_path).convert('RGB')
    ax.imshow(img)
    for box in result.boxes:
        x1,y1,x2,y2 = box.xyxy[0].tolist()
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,linewidth=2,edgecolor='lime',facecolor='none'))
        ax.text(x1,y1-4,f'{box.conf[0]:.2f}',color='lime',fontsize=8,bbox=dict(facecolor='black',alpha=0.4,pad=1))
    ax.set_title(fname,fontsize=7); ax.axis('off')
plt.suptitle('YOLOv8s — Test Predictions',fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'yolo_predictions.png'),dpi=150)
plt.show()

---
# Part 3 — Model 2: Faster R-CNN
Two-stage detector — Region Proposal Network followed by classification head.

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def build_frcnn():
    model = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT)
    in_f  = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_f, 2)
    return model.to(DEVICE)

if os.path.exists(FRCNN_WEIGHTS):
    print(f'Weights found — skipping training, loading {FRCNN_WEIGHTS}')
    frcnn_model = build_frcnn()
    frcnn_model.load_state_dict(torch.load(FRCNN_WEIGHTS, map_location=DEVICE, weights_only=True))
else:
    print('No weights found — training Faster R-CNN...')
    frcnn_model = build_frcnn()
    optimizer = torch.optim.SGD([p for p in frcnn_model.parameters() if p.requires_grad],
                                 lr=0.005, momentum=0.9, weight_decay=0.0005)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    best_val  = float('inf')

    for epoch in range(1, 31):
        t0 = time.time()
        frcnn_model.train(); total_train=0
        for images,targets in tqdm(train_loader,desc=f'FRCNN E{epoch} train',leave=False):
            images=[img.to(DEVICE) for img in images]
            targets=[{k:v.to(DEVICE) for k,v in t.items()} for t in targets]
            loss=sum(frcnn_model(images,targets).values())
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_train+=loss.item()

        frcnn_model.train(); total_val=0
        with torch.no_grad():
            for images,targets in tqdm(val_loader,desc=f'FRCNN E{epoch} val  ',leave=False):
                images=[img.to(DEVICE) for img in images]
                targets=[{k:v.to(DEVICE) for k,v in t.items()} for t in targets]
                total_val+=sum(frcnn_model(images,targets).values()).item()

        train_loss=total_train/len(train_loader); val_loss=total_val/len(val_loader)
        scheduler.step()
        flag=''
        if val_loss < best_val:
            best_val=val_loss; torch.save(frcnn_model.state_dict(),FRCNN_WEIGHTS); flag=' <- best'
        print(f'Epoch {epoch:2d}/30 | Train: {train_loss:.4f} | Val: {val_loss:.4f} | {time.time()-t0:.0f}s{flag}')

    print('Training complete!')

frcnn_model.eval()
print('Faster R-CNN ready.')

### Faster R-CNN — Evaluate on Test Set

In [ ]:
print('Finding exact best Faster R-CNN threshold on validation scores...')
frcnn_val_cache = collect_predictions(frcnn_model, val_loader, DEVICE)
best_frcnn_thresh, frcnn_val_metrics = find_best_threshold(frcnn_val_cache)
print(f'Best validation F1 threshold: {best_frcnn_thresh:.4f}')
print(f'Val P={frcnn_val_metrics[0]:.4f} R={frcnn_val_metrics[1]:.4f} F1={frcnn_val_metrics[2]:.4f}')

frcnn_test_cache = collect_predictions(frcnn_model, test_loader, DEVICE)
frcnn_p, frcnn_r, frcnn_f1, frcnn_iou = evaluate_cached(frcnn_test_cache, best_frcnn_thresh)
print(f'Precision : {frcnn_p:.4f}')
print(f'Recall    : {frcnn_r:.4f}')
print(f'F1        : {frcnn_f1:.4f}')
print(f'Mean IoU  : {frcnn_iou:.4f}')
with open(os.path.join(RESULTS_DIR,'fasterrcnn_metrics.json'),'w') as f:
    json.dump({'model':'Faster R-CNN (ResNet50-FPN v2)','threshold':round(float(best_frcnn_thresh),4),
               'precision':round(frcnn_p,4),'recall':round(frcnn_r,4),
               'f1':round(frcnn_f1,4),'mean_iou':round(frcnn_iou,4)},f,indent=2)

### Faster R-CNN — Sample Predictions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8)); axes=axes.flatten()
with torch.no_grad():
    for ax, idx in zip(axes, random.sample(range(len(test_ds)),8)):
        img_tensor,target = test_ds[idx]
        pred = frcnn_model([img_tensor.to(DEVICE)])[0]
        ax.imshow(img_tensor.permute(1,2,0).numpy())
        for box in target['boxes']:
            x1,y1,x2,y2=box.tolist()
            ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,linewidth=2,edgecolor='red',facecolor='none'))
        for box,score in zip(pred['boxes'],pred['scores']):
            if score<best_frcnn_thresh: continue
            x1,y1,x2,y2=box.cpu().tolist()
            ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,linewidth=2,edgecolor='lime',facecolor='none'))
            ax.text(x1,y1-4,f'{score:.2f}',color='lime',fontsize=8,bbox=dict(facecolor='black',alpha=0.4,pad=1))
        ax.axis('off')
plt.suptitle('Faster R-CNN — Test Predictions (red=GT, lime=pred)',fontsize=12)
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR,'fasterrcnn_predictions.png'),dpi=150); plt.show()

---
# Part 4 — Model 3: RetinaNet
Single-stage detector with focal loss and two-phase training (freeze backbone → fine-tune all).

In [ ]:
from torchvision.models.detection import retinanet_resnet50_fpn_v2, RetinaNet_ResNet50_FPN_V2_Weights
from torchvision.models.detection.retinanet import RetinaNetClassificationHead

def build_retinanet():
    model = retinanet_resnet50_fpn_v2(weights=RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT)
    num_anchors = model.head.classification_head.num_anchors
    in_channels = model.head.classification_head.conv[0][0].in_channels
    model.head.classification_head = RetinaNetClassificationHead(
        in_channels=in_channels, num_anchors=num_anchors, num_classes=2,
        norm_layer=torch.nn.BatchNorm2d)
    return model.to(DEVICE)

def eval_f1(model, loader, threshold=0.45):
    model.eval(); tp,fp,fn=0,0,0
    with torch.no_grad():
        for images,targets in loader:
            images=[img.to(DEVICE) for img in images]; preds=model(images)
            for pred,target in zip(preds,targets):
                gt_boxes=target['boxes'].numpy()
                pred_boxes=pred['boxes'][pred['scores']>=threshold].cpu().numpy()
                matched=set()
                for pb in pred_boxes:
                    best_iou,best_j=0,-1
                    for j,gb in enumerate(gt_boxes):
                        iou=compute_iou(pb,gb)
                        if iou>best_iou: best_iou,best_j=iou,j
                    if best_iou>=0.5 and best_j not in matched: tp+=1; matched.add(best_j)
                    else: fp+=1
                fn+=len(gt_boxes)-len(matched)
    p=tp/(tp+fp+1e-6); r=tp/(tp+fn+1e-6)
    return 2*p*r/(p+r+1e-6)

def run_phase(model, lr, max_epochs, patience, phase_name, use_cosine=False):
    params=[p for p in model.parameters() if p.requires_grad]
    optimizer=torch.optim.AdamW(params,lr=lr,weight_decay=0.0005)
    scheduler=(torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=max_epochs,eta_min=lr/20)
               if use_cosine else
               torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='min',factor=0.5,patience=3))
    best_f1=0.0; no_improve=0
    for epoch in range(1,max_epochs+1):
        t0=time.time(); model.train(); total=0
        for images,targets in tqdm(train_loader,desc=f'[{phase_name}] E{epoch}',leave=False):
            images=[img.to(DEVICE) for img in images]
            targets=[{k:v.to(DEVICE) for k,v in t.items()} for t in targets]
            loss=sum(model(images,targets).values())
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
            optimizer.step(); total+=loss.item()
        train_loss=total/len(train_loader)
        val_f1=eval_f1(model,val_loader)
        if use_cosine: scheduler.step()
        else: scheduler.step(train_loss)
        flag=''
        if val_f1>best_f1:
            best_f1=val_f1; no_improve=0
            torch.save(model.state_dict(),RETINA_WEIGHTS); flag=' <- best'
        else: no_improve+=1
        print(f'[{phase_name}] Epoch {epoch:2d}/{max_epochs} | Train: {train_loss:.4f} | Val F1: {val_f1:.4f} | {time.time()-t0:.0f}s{flag}')
        if no_improve>=patience: print(f'Early stopping at epoch {epoch}'); break

if os.path.exists(RETINA_WEIGHTS):
    print(f'Weights found — skipping training, loading {RETINA_WEIGHTS}')
    retina_model = build_retinanet()
    retina_model.load_state_dict(torch.load(RETINA_WEIGHTS, map_location=DEVICE, weights_only=True))
else:
    print('No weights found — training RetinaNet (two phases)...')
    retina_model = build_retinanet()
    print('--- Phase 1: Head only ---')
    for p in retina_model.backbone.parameters(): p.requires_grad=False
    run_phase(retina_model, lr=1e-3, max_epochs=40, patience=10, phase_name='Phase1')
    print('--- Phase 2: Full fine-tune ---')
    for p in retina_model.backbone.parameters(): p.requires_grad=True
    retina_model.load_state_dict(torch.load(RETINA_WEIGHTS,map_location=DEVICE, weights_only=True))
    run_phase(retina_model, lr=1e-4, max_epochs=80, patience=15, phase_name='Phase2', use_cosine=True)
    print('Training complete!')

retina_model.eval()
print('RetinaNet ready.')

### RetinaNet — Find Best Threshold & Evaluate on Test Set

In [ ]:
print('Finding exact best RetinaNet threshold on validation scores...')
retina_val_cache = collect_predictions(retina_model, val_loader, DEVICE)
best_thresh, retina_val_metrics = find_best_threshold(retina_val_cache)
print(f'Best validation F1 threshold: {best_thresh:.4f}')
print(f'Val P={retina_val_metrics[0]:.4f} R={retina_val_metrics[1]:.4f} F1={retina_val_metrics[2]:.4f}')

retina_test_cache = collect_predictions(retina_model, test_loader, DEVICE)
retina_p,retina_r,retina_f1,retina_iou = evaluate_cached(retina_test_cache, best_thresh)
print(f'Precision : {retina_p:.4f}')
print(f'Recall    : {retina_r:.4f}')
print(f'F1        : {retina_f1:.4f}')
print(f'Mean IoU  : {retina_iou:.4f}')
with open(os.path.join(RESULTS_DIR,'retinanet_metrics.json'),'w') as f:
    json.dump({'model':'RetinaNet (ResNet50-FPN v2)','threshold':round(float(best_thresh),4),
               'precision':round(retina_p,4),'recall':round(retina_r,4),
               'f1':round(retina_f1,4),'mean_iou':round(retina_iou,4)},f,indent=2)

### RetinaNet — Sample Predictions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8)); axes=axes.flatten()
with torch.no_grad():
    for ax, idx in zip(axes, random.sample(range(len(test_ds)),8)):
        img_tensor,target = test_ds[idx]
        pred = retina_model([img_tensor.to(DEVICE)])[0]
        ax.imshow(img_tensor.permute(1,2,0).numpy())
        for box in target['boxes']:
            x1,y1,x2,y2=box.tolist()
            ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,linewidth=2,edgecolor='red',facecolor='none'))
        for box,score in zip(pred['boxes'],pred['scores']):
            if score<best_thresh: continue
            x1,y1,x2,y2=box.cpu().tolist()
            ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,linewidth=2,edgecolor='lime',facecolor='none'))
            ax.text(x1,y1-4,f'{score:.2f}',color='lime',fontsize=8,bbox=dict(facecolor='black',alpha=0.4,pad=1))
        ax.axis('off')
plt.suptitle('RetinaNet — Test Predictions (red=GT, lime=pred)',fontsize=12)
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR,'retinanet_predictions.png'),dpi=150); plt.show()

---
# Part 5 — Final Comparison
Side-by-side results across all three models.

In [ ]:
print('\nYOLOv8s — Ultralytics detector metrics')
print('='*72)
print(f'Precision: {yolo_precision:.4f} | Recall: {yolo_recall:.4f} | F1: {yolo_f1:.4f}')
print(f'mAP@0.5 : {yolo_map50:.4f} | mAP@0.5:0.95: {yolo_map50_95:.4f}')

print('\nTorchvision models — custom IoU>=0.5, validation-tuned threshold')
print('='*72)
print(f'  {"Model":<25} {"Threshold":>9} {"Precision":>9} {"Recall":>9} {"F1":>9} {"Mean IoU":>9}')
torchvision_results = [
    ('RetinaNet',     best_thresh,        retina_p, retina_r, retina_f1, retina_iou),
    ('Faster R-CNN',  best_frcnn_thresh,  frcnn_p,  frcnn_r,  frcnn_f1,  frcnn_iou),
]
torchvision_results.sort(key=lambda x: x[4], reverse=True)
for name,th,p,r,f,miou in torchvision_results:
    print(f'  {name:<25} {th:>9.2f} {p:>9.4f} {r:>9.4f} {f:>9.4f} {miou:>9.4f}')
print('\nNote: YOLO mAP comes from Ultralytics; torchvision F1 is a custom greedy-match metric, so treat cross-family comparisons as approximate.')

### Comparison Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# YOLO is shown separately because its metrics come from the Ultralytics evaluator.
yolo_labels = ['Precision', 'Recall', 'F1', 'mAP@0.5', 'mAP@0.5:0.95']
yolo_vals = [yolo_precision, yolo_recall, yolo_f1, yolo_map50, yolo_map50_95]
bars = axes[0].bar(yolo_labels, yolo_vals, color='#4C9BE8', alpha=0.85)
for bar, val in zip(bars, yolo_vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=8)
axes[0].set_ylim(0, 1.05)
axes[0].set_title('YOLOv8s — Ultralytics Metrics')
axes[0].set_ylabel('Score')
axes[0].tick_params(axis='x', rotation=25)

labels = ['Precision', 'Recall', 'F1', 'Mean IoU']
retina_vals = [retina_p, retina_r, retina_f1, retina_iou]
frcnn_vals = [frcnn_p, frcnn_r, frcnn_f1, frcnn_iou]
x, width = np.arange(len(labels)), 0.35
bars1 = axes[1].bar(x - width/2, retina_vals, width, label=f'RetinaNet @ {best_thresh:.2f}', color='#6BCB77', alpha=0.85)
bars2 = axes[1].bar(x + width/2, frcnn_vals, width, label=f'Faster R-CNN @ {best_frcnn_thresh:.2f}', color='#E87B4C', alpha=0.85)
for bar in list(bars1) + list(bars2):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels)
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Torchvision Models — Custom IoU>=0.5 F1')
axes[1].legend(fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Evaluation Summary — Metrics Shown by Evaluator Family', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'model_comparison.png'),dpi=150)
plt.show()

### Side-by-Side Predictions (same images, all 3 models)

In [ ]:
sample_indices = random.sample(range(len(test_ds)), 4)
fig, axes = plt.subplots(4, 3, figsize=(16, 16))

with torch.no_grad():
    for row, idx in enumerate(sample_indices):
        img_tensor, target = test_ds[idx]
        tensor_in = [img_tensor.to(DEVICE)]
        preds = {
            'YOLOv8s':      None,
            'Faster R-CNN': frcnn_model(tensor_in)[0],
            'RetinaNet':    retina_model(tensor_in)[0],
        }
        img_path = test_ds.samples[idx][0]
        yolo_pred = yolo_model.predict(img_path, conf=0.25, verbose=False)[0]

        for col, (model_name, pred) in enumerate(preds.items()):
            ax = axes[row][col]
            ax.imshow(img_tensor.permute(1,2,0).numpy())
            for box in target['boxes']:
                x1,y1,x2,y2=box.tolist()
                ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,linewidth=2,edgecolor='red',facecolor='none'))
            if model_name == 'YOLOv8s':
                for box in yolo_pred.boxes:
                    x1,y1,x2,y2=box.xyxy[0].tolist()
                    ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,linewidth=2,edgecolor='lime',facecolor='none'))
                    ax.text(x1,y1-4,f'{box.conf[0]:.2f}',color='lime',fontsize=8,bbox=dict(facecolor='black',alpha=0.4,pad=1))
            else:
                thresh = best_thresh if model_name=='RetinaNet' else best_frcnn_thresh
                for box,score in zip(pred['boxes'],pred['scores']):
                    if score<thresh: continue
                    x1,y1,x2,y2=box.cpu().tolist()
                    ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,linewidth=2,edgecolor='lime',facecolor='none'))
                    ax.text(x1,y1-4,f'{score:.2f}',color='lime',fontsize=8,bbox=dict(facecolor='black',alpha=0.4,pad=1))
            if row==0: ax.set_title(model_name, fontsize=13, fontweight='bold')
            ax.axis('off')

plt.suptitle('Side-by-Side Predictions — red=GT, lime=predicted', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'side_by_side_comparison.png'),dpi=150)
plt.show()

---
## Download Weights & Metrics (Colab only)
Run this cell to download all trained weights and metrics JSON files to your local machine.

In [ ]:
if IN_COLAB:
    from google.colab import files
    for path in [YOLO_WEIGHTS, FRCNN_WEIGHTS, RETINA_WEIGHTS,
                 os.path.join(RESULTS_DIR,'yolo_metrics.json'),
                 os.path.join(RESULTS_DIR,'fasterrcnn_metrics.json'),
                 os.path.join(RESULTS_DIR,'retinanet_metrics.json')]:
        if os.path.exists(path):
            files.download(path)
            print(f'Downloading: {path}')
else:
    print('Running locally — weights saved in models/, metrics in results/')
    for name,path in [('YOLOv8s',YOLO_WEIGHTS),('Faster R-CNN',FRCNN_WEIGHTS),('RetinaNet',RETINA_WEIGHTS)]:
        exists = os.path.exists(path)
        print(f'  {name:<15}: {path}  [{"OK" if exists else "MISSING"}]')